# Etapa 5 — O painel cliente × mês

> **Trilha:** 1 · 2 · 3 · 4 · **5** · 6 · 7 — roteiro completo em `trilha/05_painel/README.md`

O time de CRM quer prever **quem vai comprar no mês que vem** (para agir antes de o
cliente sumir). Para isso, precisa de um **painel**: cada cliente, em cada mês.

| | |
|---|---|
| **Tabela** | `analytics.ds_cliente_mes` |
| **Grão** | 1 linha por **cliente por mês completo** — **inclusive os meses sem compra** |
| **Rótulo** | `comprou_mes_seguinte` |

A armadilha desta etapa é a mais comum de todas, e quase ninguém percebe: **o mês em que
o cliente não comprou também é um dado.**

In [ ]:
from liga import sql, executar, conferir, avaliar, progresso

## 5.1 O jeito ingênuo

O primeiro SQL que todo mundo escreve:

In [ ]:
ingenuo = sql("""
SELECT p.id AS id_pessoa,
       date_trunc('month', c.dt_criado)::date AS mes,
       count(*) AS qtd_pedidos,
       lead(date_trunc('month', c.dt_criado)::date)
           OVER (PARTITION BY p.id ORDER BY date_trunc('month', c.dt_criado)) AS proximo_mes_com_compra
FROM fato_ciclo_pedido c
JOIN dim_pessoas p ON p.sk_pessoa = c.sk_pessoa
GROUP BY p.id, date_trunc('month', c.dt_criado)
""")
print("linhas:", len(ingenuo))
ingenuo[ingenuo.id_pessoa == 10]

Três problemas, todos de modelagem:

1. **Faltam linhas.** O `GROUP BY` só cria linha para os meses **com** compra. Os meses em
   que o cliente não comprou não viram linha nenhuma.
2. **O "mês seguinte" está errado.** O `LEAD` olha a próxima **linha**, que é o próximo mês
   **com compra**. O cliente 10 comprou em maio e só voltou em agosto: para a tabela
   ingênua, o "seguinte" de maio é agosto. Junho e julho, os meses em que ele sumiu —
   exatamente o que o CRM quer prever — **não existem**.
3. **Meses pela metade.** Veja abaixo.

## 5.2 Meses completos

In [ ]:
sql("""
SELECT date_trunc('month', dt_criado)::date AS mes,
       min(dt_criado)::date                 AS primeiro_dia_com_dado,
       max(dt_criado)::date                 AS ultimo_dia_com_dado,
       count(*)                             AS pedidos
FROM fato_ciclo_pedido
GROUP BY 1
ORDER BY 1
""")

O DW começa em **5 de março** e termina em **1º de setembro**. Março e setembro estão
incompletos: comparar o "março" de um cliente com o "maio" dele é comparar 27 dias com
31 — e o "setembro" tem um dia só. O painel usa **só meses completos**: de abril a agosto.

## 5.3 A grade

A solução é a mesma da etapa 3: primeiro a **grade** (todo cliente × todo mês completo),
depois um `LEFT JOIN` das compras, com `0` onde não houve compra.

In [ ]:
sql("""
SELECT (SELECT count(DISTINCT id) FROM dim_pessoas) AS clientes,
       5                                           AS meses_completos,
       (SELECT count(DISTINCT id) FROM dim_pessoas) * 5 AS linhas_esperadas
""")

**2.500 linhas.** Com a grade, o `LEAD` passa a olhar o mês seguinte **do calendário**, e
o último mês (agosto) fica com o rótulo **nulo**: setembro ainda não aconteceu.

Duas decisões de negócio também estão no SQL, e vale lê-las:

* pedidos **cancelados ou devolvidos não contam** como compra;
* `qtd_pedidos_3m` soma o mês e os **dois anteriores** — uma janela que olha para trás,
  nunca para a frente.

## 5.4 Construindo a tabela

In [ ]:
executar("""
DROP TABLE IF EXISTS analytics.ds_cliente_mes;

CREATE TABLE analytics.ds_cliente_mes AS
WITH pedidos AS (
    SELECT p.id AS id_pessoa, c.dt_criado, c.valor_total, c.status_atual
    FROM public.fato_ciclo_pedido c
    JOIN public.dim_pessoas p ON p.sk_pessoa = c.sk_pessoa
),
periodo AS (
    SELECT min(dt_criado)::date AS inicio, max(dt_criado)::date AS fim FROM pedidos
),
meses AS (
    -- so meses COMPLETOS: o DW comeca no meio de um mes e termina no 1o dia de outro
    SELECT m::date AS mes
    FROM periodo,
         generate_series(date_trunc('month', inicio), date_trunc('month', fim), interval '1 month') AS m
    WHERE m::date >= inicio
      AND (m + interval '1 month' - interval '1 day')::date <= fim
),
clientes AS (
    SELECT id AS id_pessoa, min(created_at)::date AS dt_cadastro
    FROM public.dim_pessoas
    GROUP BY id
),
grade AS (
    -- a GRADE: todo cliente, em todo mes em que ja era cliente - comprando ou nao
    SELECT c.id_pessoa, c.dt_cadastro, m.mes
    FROM clientes c
    JOIN meses m ON c.dt_cadastro < m.mes
),
compras AS (
    SELECT id_pessoa, date_trunc('month', dt_criado)::date AS mes,
           count(*) AS qtd_pedidos, sum(valor_total) AS valor_total
    FROM pedidos
    WHERE status_atual NOT IN ('cancelado', 'devolvido')
    GROUP BY 1, 2
),
mensal AS (
    SELECT g.id_pessoa, g.dt_cadastro, g.mes,
           COALESCE(c.qtd_pedidos, 0)::integer AS qtd_pedidos,
           COALESCE(c.valor_total, 0)          AS valor_total
    FROM grade g
    LEFT JOIN compras c ON c.id_pessoa = g.id_pessoa AND c.mes = g.mes
)
SELECT id_pessoa,
       mes,
       qtd_pedidos,
       valor_total,
       ((extract(year FROM mes) - extract(year FROM dt_cadastro)) * 12
         + extract(month FROM mes) - extract(month FROM dt_cadastro))::integer AS meses_de_cliente,
       sum(qtd_pedidos) OVER (PARTITION BY id_pessoa ORDER BY mes
                              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)::integer AS qtd_pedidos_3m,
       -- o ROTULO: o mes SEGUINTE. Nulo no ultimo mes, porque ainda nao aconteceu
       CASE WHEN lead(mes) OVER (PARTITION BY id_pessoa ORDER BY mes) IS NOT NULL
            THEN lead(qtd_pedidos) OVER (PARTITION BY id_pessoa ORDER BY mes) > 0
       END AS comprou_mes_seguinte
FROM mensal;

ALTER TABLE analytics.ds_cliente_mes ADD PRIMARY KEY (id_pessoa, mes);
""")

In [ ]:
conferir("ds_cliente_mes")

## 5.5 O painel em uso

Uma pergunta que o painel responde e a tabela ingênua não conseguiria: quem não comprou
nos últimos 3 meses volta a comprar?

In [ ]:
painel = sql("SELECT * FROM analytics.ds_cliente_mes WHERE comprou_mes_seguinte IS NOT NULL")
(painel.assign(faixa=painel.qtd_pedidos_3m.clip(upper=4))
       .groupby("faixa").comprou_mes_seguinte.agg(["count", "mean"])
       .rename(columns={"count": "clientes_mes", "mean": "taxa_de_recompra"})
       .round(2))

A linha `faixa = 0` (clientes que **não compraram** nos últimos 3 meses) só existe porque
a grade criou os meses sem compra. Na tabela ingênua, esses 154 casos simplesmente não
apareceriam — e um modelo de churn nunca veria um cliente sumindo.

(Neste universo simulado as compras são sorteadas sem "cansaço" do cliente, por isso a
taxa de recompra é parecida em todas as faixas. O ponto desta etapa não é a taxa: é a
linha **existir**.)

In [ ]:
progresso()

✅ **Etapa 5 concluída.**

➡️ Continue em **`06_serie_temporal.ipynb`**.